<a href="https://colab.research.google.com/github/hamed85/ML-challenges/blob/main/spark_Practice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, expr

In [ ]:
spark = SparkSession.builder.appName("Ecommerce Analysis").getOrCreate()

In [ ]:
# df = spark.read.csv("hdfs:///user_data/online_retail.csv", header=True, inferSchema=True)
# another option to read from CSV locally
df = spark.read.csv("online_retail.csv", header=True, inferSchema=True)

In [ ]:
df.printSchema()
df.show(5)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|

In [ ]:
# Clean Data (remove null or invalid entries)
df_clean = df.dropna(subset=["Country", "Quantity", "UnitPrice"])

In [ ]:
# Task 5:
transactions = df_clean.groupBy("Country").agg(count("*").alias("TotalTransactions"))

transactions.show(5)

+---------+-----------------+
|  Country|TotalTransactions|
+---------+-----------------+
|   Sweden|              462|
|Singapore|              229|
|  Germany|             9495|
|   France|             8557|
|   Greece|              146|
+---------+-----------------+
only showing top 5 rows


In [ ]:
# Task 6:
results = df_clean.groupBy("Country").agg(
    count("*").alias("TotalTransactions"),
    _sum("Quantity").alias("TotalQuantity"),
    _sum(expr("Quantity * UnitPrice")).alias("TotalRevenue")
)

results.show(5)

+---------+-----------------+-------------+------------------+
|  Country|TotalTransactions|TotalQuantity|      TotalRevenue|
+---------+-----------------+-------------+------------------+
|   Sweden|              462|        35637| 36595.90999999998|
|Singapore|              229|         5234| 9120.390000000001|
|  Germany|             9495|       117448|221698.21000000037|
|   France|             8557|       110480|197403.90000000037|
|   Greece|              146|         1556|4710.5199999999995|
+---------+-----------------+-------------+------------------+
only showing top 5 rows


In [ ]:
top_countries = results.orderBy(col("TotalRevenue").desc())
top_countries.show(10)

+--------------+-----------------+-------------+------------------+
|       Country|TotalTransactions|TotalQuantity|      TotalRevenue|
+--------------+-----------------+-------------+------------------+
|United Kingdom|           495478|      4263829| 8187806.363998723|
|   Netherlands|             2371|       200128| 284661.5399999992|
|          EIRE|             8196|       142637| 263276.8199999992|
|       Germany|             9495|       117448|221698.21000000037|
|        France|             8557|       110480|197403.90000000037|
|     Australia|             1259|        83653|137077.26999999987|
|   Switzerland|             2002|        30325| 56385.35000000011|
|         Spain|             2533|        26824| 54774.58000000016|
|       Belgium|             2069|        23152|40910.960000000014|
|        Sweden|              462|        35637| 36595.90999999998|
+--------------+-----------------+-------------+------------------+
only showing top 10 rows


In [ ]:
#top_countries.write.mode("overwrite").csv("hdfs:///user_data/output/task6",
    #"hdfs:///user/<your-group>/ecommerce/output/spark_summary",
   # header=True)

# Stop Spark Session
#spark.stop()

## Task 7 (FPGrowth)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import collect_set
from pyspark.ml.fpm import FPGrowth

spark = SparkSession.builder.appName("E-Commerce FPGrowth").getOrCreate()

In [ ]:
#df = spark.read.csv("hdfs:///user_data/online_retail.csv", header=True, inferSchema=True)

In [ ]:
baskets = df.select("InvoiceNo", "StockCode") \
    .dropna(subset=["InvoiceNo", "StockCode"]) \
    .groupBy("InvoiceNo") \
    .agg(collect_set("StockCode").alias("items"))

In [ ]:
fpGrowth = FPGrowth(itemsCol="items", minSupport=0.02, minConfidence=0.3)
model = fpGrowth.fit(baskets)

In [ ]:
print("\nTop 10 Frequent Itemsets:")
model.freqItemsets.show(10, truncate=False)


Top 10 Frequent Itemsets:
+---------------+----+
|items          |freq|
+---------------+----+
|[85123A]       |2246|
|[85099B]       |2135|
|[20725]        |1608|
|[20725, 85099B]|588 |
|[22720]        |1462|
|[21212]        |1334|
|[20727]        |1295|
|[20727, 20725] |648 |
|[20727, 22383] |587 |
|[POST]         |1254|
+---------------+----+
only showing top 10 rows


In [ ]:
print("\nTop 10 Association Rules:")
model.associationRules.orderBy("lift", ascending=False).show(10, truncate=False)


Top 10 Association Rules:
+--------------+----------+------------------+------------------+--------------------+
|antecedent    |consequent|confidence        |lift              |support             |
+--------------+----------+------------------+------------------+--------------------+
|[22697, 22699]|[22698]   |0.7002551020408163|22.614223370146064|0.021196911196911198|
|[22698, 22699]|[22697]   |0.8941368078175895|21.909312509437626|0.021196911196911198|
|[23301]       |[23300]   |0.5941558441558441|20.11586452762923 |0.021196911196911198|
|[23300]       |[23301]   |0.7176470588235294|20.11586452762923 |0.021196911196911198|
|[22698, 22697]|[22699]   |0.8524844720496895|19.71370341614907 |0.021196911196911198|
|[22697]       |[22698]   |0.609271523178808 |19.67597562385427 |0.024864864864864864|
|[22698]       |[22697]   |0.8029925187032418|19.67597562385427 |0.024864864864864864|
|[22630]       |[22629]   |0.6255813953488372|17.844227025919476|0.020772200772200773|
|[22629]       |

In [ ]:
print(f"Number of partitions used: {baskets.rdd.getNumPartitions()}")

Number of partitions used: 2


In [ ]:
spark.stop()